# Transformer Baseline Notebook
This notebook implements and trains the baseline encoder model under our unified, leakage-safe LOBench replication pipeline.

In [1]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

Mounted at /content/drive
Mounted Google Drive and changed directory to baselines.


In [2]:
# Install PyTorch Lightning if it is not present in the environment
!pip install -q lightning pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 58.7 MB/s eta 0:00:00


In [3]:
from common import *
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Libraries and common module imported successfully.')

Libraries and common module imported successfully.


In [4]:
class TransformerEncoder(nn.Module):
    def __init__(self, n_features=40, d_model=128, nhead=4, num_layers=3,
                 dim_feedforward=256, latent_dim=256, seq_len=100):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.proj = nn.Linear(d_model, latent_dim)

        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward(self, x):  # x: [B, 100, 40]
        x = self.input_proj(x) + self.pos_embedding    # [B, 100, d_model]
        x = self.transformer(x)                         # [B, 100, d_model]
        pooled = x.mean(dim=1)                           # mean-pool over time
        return self.proj(pooled)

In [5]:
model_name = 'Transformer'
stocks = ['sz000001', 'sz000002', 'sz000858', 'sz300147', 'sz002415']
for stock in stocks:
    print(f'\n========================================')
    print(f'Starting experiment for Model: {model_name} | Stock: {stock}')
    print(f'========================================')
    run_experiment(
        encoder_class=TransformerEncoder,
        model_name=model_name,
        stock=stock,
        latent_dim=256,
        max_epochs=100
    )


Starting experiment for Model: Transformer | Stock: sz000001
Loading data from data/sz000001-level10_processed.csv...
Loaded shape: (1171534, 41)
Columns and Level 10 ordering verified successfully.
Detecting sessions (3-second continuous chunks)...
Detected 488 sessions.
Splitting train/val/test splits (80/10/10 by calendar dates)...
Constructing sequence window indices (seq_len=100, stride=1)...

--- Train/Val/Test Splits & Windows ---
Split      | Rows       | Window Count
Train      | 936254     | 897644
Validation | 115231     | 110479
Test       | 120049     | 115099
---------------------------------------

Encoder parameters: 448,512
Shared Decoder parameters: 4,756,896
Total model parameters: 5,205,408


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Found existing checkpoint at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/Transformer/sz000001/last-v5.ckpt. Resuming training...


/usr/local/lib/python3.12/dist-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory training_logs/Transformer/sz000001 exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/Transformer/sz000001 exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/Transformer/sz000001/last-v5.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/Transformer/sz000001/last-v5.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ TransformerEncoder │  448 K │ train │     0 │
│ 1 │ decoder │ SharedDecoder      │  4.8 M │ train │     0 │
│ 2 │ loss_fn │ MSELoss            │      0 │ train │     0 │
│ 3 │ mae_fn  │ L1Loss             │      0 │ train │     0 │
└───┴─────────┴────────────────────┴────────┴───────┴───────┘

...

Training loop finished in 6240 seconds.
Loading best checkpoint for evaluation: /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/Transformer/sz002415/best.ckpt


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │    0.10679683834314346    │
│         test_mse          │    0.06734336167573929    │
└───────────────────────────┴───────────────────────────┘


=== FINAL REPORT ===
Model: Transformer | Stock: sz002415
Encoder params: 448,512
Total params (encoder + shared decoder): 5,205,408
Training time: 6240s
Test MSE: 0.0673
Test MAE: 0.1068

